# Advanced Fraud Detection Visualization Suite

This notebook contains comprehensive visualizations for analyzing online payment fraud patterns and model performance evaluation. The visualizations are designed to meet IEEE publication standards.

## Table of Contents
1. Data Loading and Initial Setup
2. Transaction Distribution Analysis
3. Fraud Pattern Visualization
4. Balance Change Analysis
5. Model Performance Visualization
6. Fraud Alert System Dashboard

## 1. Data Loading and Initial Setup
Import required libraries and configure plotting settings for IEEE paper quality visualizations.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Configure plotting for IEEE paper style
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

# Load the dataset
df = pd.read_csv('payment_fraud.csv')

## 2. Transaction Distribution Analysis
Analyze and visualize the distribution of transactions across different types and amounts.

In [ ]:
def plot_transaction_distributions():
    # Create a figure with subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Transaction Type Distribution',
            'Transaction Amount Distribution',
            'Hourly Transaction Pattern',
            'Transaction Volume by Type'
        )
    )
    
    # 1. Transaction Type Distribution
    type_counts = df['type'].value_counts()
    fig.add_trace(
        go.Bar(
            x=type_counts.index,
            y=type_counts.values,
            name='Transaction Types',
            marker_color='rgb(55, 83, 109)'
        ),
        row=1, col=1
    )
    
    # 2. Amount Distribution
    fig.add_trace(
        go.Histogram(
            x=df['amount'],
            nbinsx=50,
            name='Amount Distribution',
            marker_color='rgb(26, 118, 255)'
        ),
        row=1, col=2
    )
    
    # 3. Hourly Pattern
    hourly_trans = df.groupby('step').size()
    fig.add_trace(
        go.Scatter(
            x=hourly_trans.index,
            y=hourly_trans.values,
            mode='lines+markers',
            name='Hourly Pattern',
            line=dict(color='rgb(76, 175, 80)', width=2)
        ),
        row=2, col=1
    )
    
    # 4. Volume by Type
    volume_by_type = df.groupby('type')['amount'].sum()
    fig.add_trace(
        go.Bar(
            x=volume_by_type.index,
            y=volume_by_type.values,
            name='Transaction Volume',
            marker_color='rgb(158, 202, 225)'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text='Multi-dimensional Transaction Analysis',
        showlegend=True,
        height=1000,
        width=1200,
        template='plotly_white'
    )
    
    fig.show()

# Generate the visualization
plot_transaction_distributions()

## 3. Fraud Pattern Visualization
Analyze and visualize patterns specific to fraudulent transactions.

In [ ]:
def visualize_fraud_patterns():
    # Create figure with secondary y-axis
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Fraud Distribution by Transaction Type',
            'Amount Distribution: Fraud vs Normal',
            'Temporal Fraud Patterns',
            'Fraud Risk Heatmap'
        )
    )
    
    # 1. Fraud Distribution by Type
    fraud_by_type = pd.crosstab(df['type'], df['isFraud'])
    fig.add_trace(
        go.Bar(
            x=fraud_by_type.index,
            y=fraud_by_type[1],
            name='Fraudulent',
            marker_color='red'
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(
            x=fraud_by_type.index,
            y=fraud_by_type[0],
            name='Normal',
            marker_color='blue'
        ),
        row=1, col=1
    )
    
    # 2. Amount Distribution
    fig.add_trace(
        go.Violin(
            x=df[df['isFraud'] == 0]['amount'],
            name='Normal Transactions',
            side='negative',
            line_color='blue'
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Violin(
            x=df[df['isFraud'] == 1]['amount'],
            name='Fraudulent Transactions',
            side='positive',
            line_color='red'
        ),
        row=1, col=2
    )
    
    # 3. Temporal Patterns
    fraud_by_time = df[df['isFraud'] == 1].groupby('step').size()
    fig.add_trace(
        go.Scatter(
            x=fraud_by_time.index,
            y=fraud_by_time.values,
            mode='lines+markers',
            name='Fraud Occurrences',
            line=dict(color='red', width=2)
        ),
        row=2, col=1
    )
    
    # 4. Risk Heatmap
    risk_matrix = pd.crosstab(
        pd.qcut(df['amount'], 4),
        pd.qcut(df['step'], 4),
        values=df['isFraud'],
        aggfunc='mean'
    )
    
    fig.add_trace(
        go.Heatmap(
            z=risk_matrix.values,
            x=risk_matrix.columns.astype(str),
            y=risk_matrix.index.astype(str),
            colorscale='RdYlBu_r',
            name='Fraud Risk'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text='Comprehensive Fraud Pattern Analysis',
        height=1000,
        width=1200,
        template='plotly_white'
    )
    
    fig.show()

# Generate the visualization
visualize_fraud_patterns()

## 4. Balance Change Analysis
Analyze how account balances change in fraudulent vs legitimate transactions.

In [ ]:
def analyze_balance_changes():
    # Calculate balance changes
    df['oldBalanceChange'] = df['newbalanceOrig'] - df['oldbalanceOrg']
    df['destBalanceChange'] = df['newbalanceDest'] - df['oldbalanceDest']
    
    # Create figure
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Origin Balance Changes',
            'Destination Balance Changes',
            'Balance Flow Analysis',
            'Balance Change Patterns'
        )
    )
    
    # 1. Origin Balance Changes
    fig.add_trace(
        go.Box(
            y=df[df['isFraud'] == 0]['oldBalanceChange'],
            name='Normal',
            marker_color='blue',
            boxpoints='outliers'
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(
            y=df[df['isFraud'] == 1]['oldBalanceChange'],
            name='Fraud',
            marker_color='red',
            boxpoints='outliers'
        ),
        row=1, col=1
    )
    
    # 2. Destination Balance Changes
    fig.add_trace(
        go.Box(
            y=df[df['isFraud'] == 0]['destBalanceChange'],
            name='Normal',
            marker_color='blue',
            boxpoints='outliers'
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Box(
            y=df[df['isFraud'] == 1]['destBalanceChange'],
            name='Fraud',
            marker_color='red',
            boxpoints='outliers'
        ),
        row=1, col=2
    )
    
    # 3. Balance Flow
    fig.add_trace(
        go.Scatter(
            x=df['oldbalanceOrg'],
            y=df['newbalanceOrig'],
            mode='markers',
            marker=dict(
                color=df['isFraud'],
                colorscale='RdBu',
                showscale=True
            ),
            name='Balance Flow'
        ),
        row=2, col=1
    )
    
    # 4. Balance Change Patterns
    fig.add_trace(
        go.Histogram2d(
            x=df['oldBalanceChange'],
            y=df['destBalanceChange'],
            colorscale='Viridis',
            nbinsx=30,
            nbinsy=30,
            name='Change Pattern'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text='Balance Change Analysis',
        height=1000,
        width=1200,
        template='plotly_white'
    )
    
    fig.show()

# Generate the visualization
analyze_balance_changes()

## 5. Model Performance Visualization
Visualize the performance metrics of the fraud detection models.

In [ ]:
def plot_model_performance(y_true, y_pred_rf, y_pred_jaya, y_pred_prob_rf, y_pred_prob_jaya):
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'ROC Curves',
            'Precision-Recall Curves',
            'Confusion Matrices',
            'Model Metrics Comparison'
        )
    )
    
    # 1. ROC Curves
    fpr_rf, tpr_rf, _ = roc_curve(y_true, y_pred_prob_rf)
    fpr_jaya, tpr_jaya, _ = roc_curve(y_true, y_pred_prob_jaya)
    
    fig.add_trace(
        go.Scatter(
            x=fpr_rf, y=tpr_rf,
            name=f'Random Forest (AUC={auc(fpr_rf, tpr_rf):.3f})',
            line=dict(color='blue', width=2)
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=fpr_jaya, y=tpr_jaya,
            name=f'Jaya (AUC={auc(fpr_jaya, tpr_jaya):.3f})',
            line=dict(color='red', width=2)
        ),
        row=1, col=1
    )
    
    # 2. Precision-Recall Curves
    precision_rf, recall_rf, _ = precision_recall_curve(y_true, y_pred_prob_rf)
    precision_jaya, recall_jaya, _ = precision_recall_curve(y_true, y_pred_prob_jaya)
    
    fig.add_trace(
        go.Scatter(
            x=recall_rf, y=precision_rf,
            name='Random Forest PR',
            line=dict(color='blue', width=2, dash='dash')
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(
            x=recall_jaya, y=precision_jaya,
            name='Jaya PR',
            line=dict(color='red', width=2, dash='dash')
        ),
        row=1, col=2
    )
    
    # 3. Confusion Matrices
    cm_rf = confusion_matrix(y_true, y_pred_rf)
    cm_jaya = confusion_matrix(y_true, y_pred_jaya)
    
    fig.add_trace(
        go.Heatmap(
            z=cm_rf,
            x=['Pred 0', 'Pred 1'],
            y=['True 0', 'True 1'],
            colorscale='Blues',
            name='RF Confusion Matrix'
        ),
        row=2, col=1
    )
    
    # 4. Metrics Comparison
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    rf_report = classification_report(y_true, y_pred_rf, output_dict=True)
    jaya_report = classification_report(y_true, y_pred_jaya, output_dict=True)
    
    rf_metrics = [rf_report['accuracy'], rf_report['macro avg']['precision'],
                 rf_report['macro avg']['recall'], rf_report['macro avg']['f1-score']]
    jaya_metrics = [jaya_report['accuracy'], jaya_report['macro avg']['precision'],
                   jaya_report['macro avg']['recall'], jaya_report['macro avg']['f1-score']]
    
    fig.add_trace(
        go.Bar(
            name='Random Forest',
            x=metrics,
            y=rf_metrics,
            marker_color='blue'
        ),
        row=2, col=2
    )
    fig.add_trace(
        go.Bar(
            name='Jaya Algorithm',
            x=metrics,
            y=jaya_metrics,
            marker_color='red'
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text='Comprehensive Model Performance Analysis',
        height=1000,
        width=1200,
        template='plotly_white'
    )
    
    fig.show()

# The function can be called with actual model predictions

## 6. Fraud Alert System Dashboard
Create an interactive dashboard for monitoring fraud alerts.

In [ ]:
def create_fraud_dashboard():
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Real-time Fraud Rate',
            'Alert Distribution',
            'Risk Level Timeline',
            'Transaction Monitor'
        ),
        specs=[[{"secondary_y": True}, {"secondary_y": True}],
               [{"secondary_y": True}, {"type": "domain"}]]
    )
    
    # 1. Real-time Fraud Rate
    time_groups = df.groupby('step')
    fraud_rate = time_groups['isFraud'].mean() * 100
    
    fig.add_trace(
        go.Scatter(
            x=fraud_rate.index,
            y=fraud_rate.values,
            mode='lines+markers',
            name='Fraud Rate (%)',
            line=dict(color='red', width=2)
        ),
        row=1, col=1
    )
    
    # 2. Alert Distribution
    alert_dist = pd.cut(df[df['isFraud'] == 1]['amount'],
                       bins=[0, 100, 1000, 10000, float('inf')],
                       labels=['Low', 'Medium', 'High', 'Critical'])
    alert_counts = alert_dist.value_counts()
    
    fig.add_trace(
        go.Bar(
            x=alert_counts.index,
            y=alert_counts.values,
            name='Alert Levels',
            marker_color=['green', 'yellow', 'orange', 'red']
        ),
        row=1, col=2
    )
    
    # 3. Risk Level Timeline
    risk_levels = pd.qcut(df['amount'], q=4, labels=['Low', 'Medium', 'High', 'Critical'])
    risk_by_time = pd.crosstab(df['step'], risk_levels)
    
    for level in risk_by_time.columns:
        fig.add_trace(
            go.Scatter(
                x=risk_by_time.index,
                y=risk_by_time[level],
                name=f'{level} Risk',
                stackgroup='one'
            ),
            row=2, col=1
        )
    
    # 4. Transaction Monitor (Pie Chart)
    status_counts = df['isFraud'].value_counts()
    fig.add_trace(
        go.Pie(
            values=status_counts.values,
            labels=['Normal', 'Fraudulent'],
            marker_colors=['green', 'red'],
            hole=0.4
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text='Real-time Fraud Detection Dashboard',
        height=1000,
        width=1200,
        showlegend=True,
        template='plotly_white'
    )
    
    fig.show()

# Generate the dashboard
create_fraud_dashboard()

# System Architecture Visualization
This section visualizes the complete architecture of our fraud detection system, including data flow and model pipeline.

In [ ]:
# Create comprehensive system architecture visualization
def create_system_architecture():
    # Create figure with secondary y-axis
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'System Architecture Flowchart',
            'Model Pipeline',
            'Data Processing Flow',
            'Evaluation Metrics'
        )
    )
    
    # 1. System Architecture Flowchart
    def add_node(x, y, name, color='lightblue'):
        return go.Scatter(
            x=[x], y=[y],
            mode='markers+text',
            text=[name],
            textposition='bottom center',
            marker=dict(size=30, symbol='square', color=color),
            showlegend=False
        )
    
    def add_arrow(x0, y0, x1, y1):
        return go.Scatter(
            x=[x0, x1],
            y=[y0, y1],
            mode='lines+markers',
            line=dict(color='black', width=1),
            marker=dict(size=6),
            showlegend=False
        )
    
    # Add nodes for main flowchart
    nodes = [
        (0, 4, 'Credit Card Fraud Dataset', 'lightgreen'),
        (0, 3, 'Preprocessing', 'lightblue'),
        (0, 2, 'Apply SMOTE', 'lightyellow'),
        (-2, 1, 'Training Set', 'lightpink'),
        (2, 1, 'Testing Set', 'lightpink'),
        (-2, 0, 'Train Models', 'lightblue'),
        (2, 0, 'Test', 'lightblue'),
        (0, -1, 'Evaluation', 'lightgreen')
    ]
    
    for x, y, name, color in nodes:
        fig.add_trace(add_node(x, y, name, color), row=1, col=1)
    
    # Add arrows
    arrows = [
        (0, 4, 0, 3),  # Dataset to Preprocessing
        (0, 3, 0, 2),  # Preprocessing to SMOTE
        (0, 2, -2, 1), # SMOTE to Training
        (0, 2, 2, 1),  # SMOTE to Testing
        (-2, 1, -2, 0), # Training to Models
        (2, 1, 2, 0),   # Testing to Test
        (-2, 0, 0, -1), # Models to Evaluation
        (2, 0, 0, -1)   # Test to Evaluation
    ]
    
    for x0, y0, x1, y1 in arrows:
        fig.add_trace(add_arrow(x0, y0, x1, y1), row=1, col=1)
    
    # 2. Model Pipeline Visualization
    models = ['Random Forest', 'Jaya Algorithm', 'ET', 'RF', 'DT', 'LR', 'AdaBoost']
    y_pos = np.arange(len(models))
    
    fig.add_trace(
        go.Bar(
            x=[1]*len(models),
            y=models,
            orientation='h',
            marker_color=['blue', 'red', 'green', 'purple', 'orange', 'cyan', 'magenta'],
            name='Models'
        ),
        row=1, col=2
    )
    
    # 3. Data Processing Flow
    processing_steps = ['Raw Data', 'Cleaning', 'Feature Engineering', 'Normalization', 'SMOTE', 'Split']
    
    for i, step in enumerate(processing_steps):
        fig.add_trace(
            go.Scatter(
                x=[i],
                y=[0],
                mode='markers+text',
                text=[step],
                textposition='bottom center',
                marker=dict(size=20, symbol='diamond', color='lightblue'),
                showlegend=False
            ),
            row=2, col=1
        )
        if i < len(processing_steps) - 1:
            fig.add_trace(
                go.Scatter(
                    x=[i, i+1],
                    y=[0, 0],
                    mode='lines+markers',
                    line=dict(color='black', width=1),
                    showlegend=False
                ),
                row=2, col=1
            )
    
    # 4. Evaluation Metrics Flow
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
    
    fig.add_trace(
        go.Scatter(
            x=[1, 2, 3, 4, 5],
            y=[1]*5,
            mode='markers+text',
            text=metrics,
            textposition='top center',
            marker=dict(size=15, symbol='circle', color='lightgreen'),
            showlegend=False
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        title_text='Comprehensive System Architecture and Workflow',
        height=1000,
        width=1200,
        showlegend=False,
        plot_bgcolor='white'
    )
    
    # Update axes
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False)
    
    fig.show()

# Generate the system architecture visualization
create_system_architecture()

# Fraud Detection System Flowchart
This section provides a clear flowchart visualization of the fraud detection pipeline.

In [ ]:
# Import required libraries
import plotly.graph_objects as go

def create_fraud_detection_flowchart():
    # Create base figure
    fig = go.Figure()
    
    # Define node positions
    nodes = {
        'dataset': {'x': 0, 'y': 5, 'text': 'Credit Card\nFraud Dataset'},
        'preprocess': {'x': 0, 'y': 4, 'text': 'Data\nPreprocessing'},
        'smote': {'x': 0, 'y': 3, 'text': 'Apply SMOTE\nBalancing'},
        'split': {'x': 0, 'y': 2, 'text': 'Train-Test\nSplit'},
        'train': {'x': -2, 'y': 1, 'text': 'Training Set'},
        'test': {'x': 2, 'y': 1, 'text': 'Testing Set'},
        'rf': {'x': -3, 'y': 0, 'text': 'Random Forest'},
        'jaya': {'x': -2, 'y': 0, 'text': 'Jaya Algorithm'},
        'et': {'x': -1, 'y': 0, 'text': 'Extra Trees'},
        'dt': {'x': -4, 'y': 0, 'text': 'Decision Tree'},
        'eval': {'x': 0, 'y': -1, 'text': 'Model\nEvaluation'},
        'deploy': {'x': 0, 'y': -2, 'text': 'Deploy Best\nModel'}
    }
    
    # Add nodes
    colors = {
        'dataset': 'lightgreen',
        'preprocess': 'lightblue',
        'smote': 'lightyellow',
        'split': 'lightpink',
        'train': 'lightcyan',
        'test': 'lightcyan',
        'rf': 'lightblue',
        'jaya': 'lightblue',
        'et': 'lightblue',
        'dt': 'lightblue',
        'eval': 'lightgreen',
        'deploy': 'lightcoral'
    }
    
    for node, pos in nodes.items():
        fig.add_trace(go.Scatter(
            x=[pos['x']], 
            y=[pos['y']],
            mode='markers+text',
            name=node,
            text=[pos['text']],
            textposition='middle center',
            textfont=dict(size=12),
            marker=dict(
                size=50,
                symbol='square',
                color=colors[node],
                line=dict(color='black', width=2)
            ),
            showlegend=False
        ))
    
    # Add arrows
    arrows = [
        ('dataset', 'preprocess'),
        ('preprocess', 'smote'),
        ('smote', 'split'),
        ('split', 'train'),
        ('split', 'test'),
        ('train', 'rf'),
        ('train', 'jaya'),
        ('train', 'et'),
        ('train', 'dt'),
        ('rf', 'eval'),
        ('jaya', 'eval'),
        ('et', 'eval'),
        ('dt', 'eval'),
        ('test', 'eval'),
        ('eval', 'deploy')
    ]
    
    for start, end in arrows:
        fig.add_trace(go.Scatter(
            x=[nodes[start]['x'], nodes[end]['x']],
            y=[nodes[start]['y'], nodes[end]['y']],
            mode='lines+markers',
            line=dict(color='black', width=1),
            marker=dict(size=8, symbol='arrow', angleref='previous'),
            showlegend=False
        ))
    
    # Add processing steps annotations
    preprocessing_steps = [
        "1. Data Cleaning",
        "2. Feature Engineering",
        "3. Normalization",
        "4. Encoding"
    ]
    
    fig.add_annotation(
        x=2,
        y=4,
        text="<br>".join(preprocessing_steps),
        showarrow=True,
        arrowhead=1,
        ax=50,
        ay=0,
        font=dict(size=10),
        align='left'
    )
    
    # Add model metrics annotation
    metrics = [
        "Metrics:",
        "• Accuracy",
        "• Precision",
        "• Recall",
        "• F1-Score",
        "• ROC-AUC"
    ]
    
    fig.add_annotation(
        x=2,
        y=-1,
        text="<br>".join(metrics),
        showarrow=True,
        arrowhead=1,
        ax=50,
        ay=0,
        font=dict(size=10),
        align='left'
    )
    
    # Update layout
    fig.update_layout(
        title={
            'text': 'Fraud Detection System Pipeline',
            'y':0.95,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': dict(size=20)
        },
        plot_bgcolor='white',
        width=1000,
        height=800,
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-5, 5]
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            showticklabels=False,
            range=[-3, 6]
        )
    )
    
    fig.show()

# Generate the flowchart
create_fraud_detection_flowchart()